# Vector NLU Playground

Interactive testing for the on-device intent + entity models.

**Setup:** `pip install tensorflow numpy matplotlib`

In [ ]:
import json
import re
import numpy as np
import tensorflow as tf
from datetime import datetime, timedelta
from training_data import INTENT_EXAMPLES, ENTITY_EXAMPLES, CATEGORY_CANONICAL, CARD_CANONICAL

OUTPUT_DIR = "output"
SEQUENCE_LENGTH = 20

def load_json(path):
    with open(path) as f:
        return json.load(f)

# --- Load intent model ---
intent_interp = tf.lite.Interpreter(model_path=f"{OUTPUT_DIR}/intent_model.tflite")
intent_interp.allocate_tensors()
intent_in = intent_interp.get_input_details()
intent_out = intent_interp.get_output_details()
intent_labels = load_json(f"{OUTPUT_DIR}/intent_labels.json")
intent_vocab = load_json(f"{OUTPUT_DIR}/intent_vocab.json")

# --- Load entity model ---
entity_interp = tf.lite.Interpreter(model_path=f"{OUTPUT_DIR}/entity_model.tflite")
entity_interp.allocate_tensors()
entity_in = entity_interp.get_input_details()
entity_out = entity_interp.get_output_details()
entity_labels = load_json(f"{OUTPUT_DIR}/entity_labels.json")
entity_vocab = load_json(f"{OUTPUT_DIR}/entity_vocab.json")

print(f"Intent labels: {intent_labels}")
print(f"Entity tags:   {entity_labels}")
print(f"Intent vocab:  {len(intent_vocab)} words")
print(f"Entity vocab:  {len(entity_vocab)} words")
print("\n✅ Models loaded")

In [ ]:
# ============================================================
# Core inference functions
# ============================================================

def text_to_seq(text, vocab, max_len=SEQUENCE_LENGTH):
    words = text.lower().split()
    seq = [vocab.get(w, 1) for w in words[:max_len]]
    seq += [0] * (max_len - len(seq))
    return np.array([seq], dtype=np.int32)


def predict_intent(text):
    """Returns (intent_name, confidence, all_probabilities)"""
    seq = text_to_seq(text, intent_vocab)
    intent_interp.set_tensor(intent_in[0]["index"], seq)
    intent_interp.invoke()
    probs = intent_interp.get_tensor(intent_out[0]["index"])[0]
    idx = np.argmax(probs)
    return intent_labels[idx], float(probs[idx]), probs


def predict_entities(text):
    """Returns (entities_dict, word_tags_list)"""
    words = text.lower().split()
    seq = text_to_seq(text, entity_vocab)
    entity_interp.set_tensor(entity_in[0]["index"], seq)
    entity_interp.invoke()
    output = entity_interp.get_tensor(entity_out[0]["index"])[0]

    entities = {}
    word_tags = []
    cur_ent = None
    cur_words = []

    for i, word in enumerate(words[:SEQUENCE_LENGTH]):
        tag_id = np.argmax(output[i])
        tag = entity_labels[tag_id]
        conf = float(output[i][tag_id])
        word_tags.append((word, tag, conf))

        if tag.startswith("B-"):
            if cur_ent and cur_words:
                entities[cur_ent.lower()] = " ".join(cur_words)
            cur_ent = tag[2:]
            cur_words = [word]
        elif tag.startswith("I-") and cur_ent == tag[2:]:
            cur_words.append(word)
        else:
            if cur_ent and cur_words:
                entities[cur_ent.lower()] = " ".join(cur_words)
            cur_ent = None
            cur_words = []

    if cur_ent and cur_words:
        entities[cur_ent.lower()] = " ".join(cur_words)

    return entities, word_tags


def resolve_date(date_str):
    """Resolve relative date strings to (from, to) date range."""
    today = datetime.now().replace(hour=0, minute=0, second=0, microsecond=0)
    fmt = lambda d: d.strftime("%Y-%m-%d")
    lower = date_str.lower().strip()

    mappings = {
        "today": (today, today),
        "yesterday": (today - timedelta(days=1), today - timedelta(days=1)),
        "this week": (today - timedelta(days=today.weekday()), today),
        "this month": (today.replace(day=1), today),
        "this year": (today.replace(month=1, day=1), today),
    }

    if lower in mappings:
        f, t = mappings[lower]
        return fmt(f), fmt(t)

    if lower == "last week":
        this_week_start = today - timedelta(days=today.weekday())
        return fmt(this_week_start - timedelta(days=7)), fmt(this_week_start - timedelta(days=1))

    if lower == "last month":
        first_this = today.replace(day=1)
        last_month_end = first_this - timedelta(days=1)
        last_month_start = last_month_end.replace(day=1)
        return fmt(last_month_start), fmt(last_month_end)

    if lower == "last year":
        return f"{today.year-1}-01-01", f"{today.year-1}-12-31"

    months = ["january","february","march","april","may","june",
              "july","august","september","october","november","december"]
    if lower in months:
        m = months.index(lower) + 1
        from calendar import monthrange
        _, last = monthrange(today.year, m)
        return f"{today.year}-{m:02d}-01", f"{today.year}-{m:02d}-{last:02d}"

    days_match = re.match(r"(\d+)\s*days?\s*ago", lower)
    if days_match:
        d = today - timedelta(days=int(days_match.group(1)))
        return fmt(d), fmt(d)

    return None


def resolve_card(card_text):
    """Resolve spoken card name to canonical issuer name."""
    cleaned = re.sub(r"\s*(credit|debit)?\s*card$", "", card_text.strip()).strip()
    return CARD_CANONICAL.get(cleaned, cleaned.upper())


def build_sql(intent, entities):
    """Generate SQL from intent + entities (mirrors mobile nlu.ts)."""
    conditions = []
    params = []

    if "merchant" in entities:
        conditions.append("description LIKE ?")
        params.append(f"%{entities['merchant']}%")

    if "category" in entities:
        canonical = CATEGORY_CANONICAL.get(entities["category"], entities["category"])
        conditions.append("category = ?")
        params.append(canonical)

    if "amount" in entities:
        numbers = re.findall(r"\d+", entities["amount"])
        if numbers:
            num = int(numbers[0])
            s = entities["amount"]
            if any(k in s for k in ["above","over","more","greater"]):
                conditions.append("amount > ?")
                params.append(num)
            elif any(k in s for k in ["below","under","less"]):
                conditions.append("amount < ?")
                params.append(num)

    if "card" in entities:
        canonical = resolve_card(entities["card"])
        # In real app this resolves to cardId UUID; here we use issuer name
        conditions.append(f"/* card: {canonical} -> cardId = ? */")

    if "date" in entities:
        resolved = resolve_date(entities["date"])
        if resolved:
            conditions.append("date BETWEEN ? AND ?")
            params.extend(resolved)

    where = " AND ".join(conditions) if conditions else "1=1"

    sql_map = {
        "count_transactions": f"SELECT COUNT(*) as result FROM transactions WHERE {where}",
        "total_spent": f"SELECT COALESCE(SUM(amount), 0) as result FROM transactions WHERE type='debit' AND {where}",
        "list_transactions": f"SELECT * FROM transactions WHERE {where} ORDER BY date DESC LIMIT 20",
        "highest_transaction": f"SELECT * FROM transactions WHERE type='debit' AND {where} ORDER BY amount DESC LIMIT 1",
        "lowest_transaction": f"SELECT * FROM transactions WHERE type='debit' AND {where} ORDER BY amount ASC LIMIT 1",
        "average_spend": f"SELECT COALESCE(AVG(amount), 0) as result FROM transactions WHERE type='debit' AND {where}",
        "category_spend": f"SELECT category, SUM(amount) as total, COUNT(*) as count FROM transactions WHERE type='debit' AND {where} GROUP BY category ORDER BY total DESC",
        "transactions_on_date": f"SELECT * FROM transactions WHERE {where} ORDER BY date DESC",
        "monthly_summary": f"SELECT strftime('%Y-%m', date) as month, SUM(amount) as total, COUNT(*) as count FROM transactions WHERE type='debit' AND {where} GROUP BY month ORDER BY month DESC",
    }

    return sql_map.get(intent, f"SELECT * FROM transactions WHERE {where}"), params


def ask(question):
    """Full pipeline: question -> intent + entities + SQL."""
    intent, conf, all_probs = predict_intent(question)
    entities, word_tags = predict_entities(question)
    sql, params = build_sql(intent, entities)

    # Display results
    print(f"\n{'─'*60}")
    print(f"  Question:   {question}")
    print(f"  Intent:     {intent}  ({conf:.1%})")

    # Show top-3 intents
    top3 = sorted(enumerate(all_probs), key=lambda x: -x[1])[:3]
    alt = "  Runners-up: " + ", ".join(f"{intent_labels[i]} ({p:.1%})" for i, p in top3[1:])
    print(alt)

    if entities:
        print(f"  Entities:   {json.dumps(entities)}")
    else:
        print(f"  Entities:   (none)")

    # Show word-level tags
    tag_str = " ".join(f"{w}[{t}]" if t != "O" else w for w, t, _ in word_tags)
    print(f"  Tags:       {tag_str}")

    print(f"  SQL:        {sql}")
    if params:
        print(f"  Params:     {params}")
    print(f"{'─'*60}")

    return {"intent": intent, "confidence": conf, "entities": entities, "sql": sql, "params": params}


print("\n✅ Pipeline ready — use ask(\"your question\") to test")

---
## Ask anything

Type your question below and run the cell. Change it and re-run as many times as you want.

In [ ]:
# ✏️ Change this and re-run
ask("how much did I spend on swiggy last month")


────────────────────────────────────────────────────────────
  Question:   how much did I spend on swiggy last month
  Intent:     total_spent  (83.9%)
  Runners-up: category_spend (11.0%), transactions_on_date (1.8%)
  Entities:   {"merchant": "swiggy", "date": "last month"}
  Tags:       how much did i spend on swiggy[B-MERCHANT] last[B-DATE] month[I-DATE]
  SQL:        SELECT COALESCE(SUM(amount), 0) as result FROM transactions WHERE type='debit' AND description LIKE ? AND date BETWEEN ? AND ?
  Params:     ['%swiggy%', '2026-02-01', '2026-02-28']
────────────────────────────────────────────────────────────


{'intent': 'total_spent',
 'confidence': 0.8385919332504272,
 'entities': {'merchant': 'swiggy', 'date': 'last month'},
 'sql': "SELECT COALESCE(SUM(amount), 0) as result FROM transactions WHERE type='debit' AND description LIKE ? AND date BETWEEN ? AND ?",
 'params': ['%swiggy%', '2026-02-01', '2026-02-28']}

In [ ]:
# ✏️ Try more questions
ask("show my food expenses this week")


────────────────────────────────────────────────────────────
  Question:   show my food expenses this week
  Intent:     list_transactions  (43.4%)
  Runners-up: transactions_on_date (24.5%), monthly_summary (8.3%)
  Entities:   {"category": "food", "date": "this week"}
  Tags:       show my food[B-CATEGORY] expenses this[B-DATE] week[I-DATE]
  SQL:        SELECT * FROM transactions WHERE category = ? AND date BETWEEN ? AND ? ORDER BY date DESC LIMIT 20
  Params:     ['Food & Dining', '2026-03-02', '2026-03-08']
────────────────────────────────────────────────────────────


{'intent': 'list_transactions',
 'confidence': 0.43426772952079773,
 'entities': {'category': 'food', 'date': 'this week'},
 'sql': 'SELECT * FROM transactions WHERE category = ? AND date BETWEEN ? AND ? ORDER BY date DESC LIMIT 20',
 'params': ['Food & Dining', '2026-03-02', '2026-03-08']}

In [ ]:
ask("total spent on zomato this month")


────────────────────────────────────────────────────────────
  Question:   what is total spends
  Intent:     total_spent  (81.0%)
  Runners-up: category_spend (15.0%), average_spend (1.9%)
  Entities:   {"date": "spends"}
  Tags:       what is total spends[B-DATE]
  SQL:        SELECT COALESCE(SUM(amount), 0) as result FROM transactions WHERE type='debit' AND 1=1
────────────────────────────────────────────────────────────


{'intent': 'total_spent',
 'confidence': 0.8102769255638123,
 'entities': {'date': 'spends'},
 'sql': "SELECT COALESCE(SUM(amount), 0) as result FROM transactions WHERE type='debit' AND 1=1",
 'params': []}

---
## Batch test

Run a batch of sample queries to get a quick feel for the model.

In [ ]:
batch_questions = [
    # Count
    "how many swiggy transactions",
    "count amazon orders last month",
    # Total spent
    "how much did I spend on food",
    "total spent on zomato this month",
    # Date
    "what did I spend yesterday",
    "show transactions from last friday",
    # Category
    "show my grocery spending",
    "entertainment expenses this month",
    # List
    "list all transactions",
    "show uber transactions",
    # Highest / Lowest
    "what was my biggest expense",
    "smallest transaction this month",
    # Average
    "average spending on uber",
    "what is my average food expense",
    # Monthly summary
    "monthly summary",
    "how was my spending this month",
    # Edge cases / harder queries
    "did I order from dominos this week",
    "how much on netflix and spotify",
    "transactions above 5000 last month",
    "total medical expenses this year",
    # Card queries
    "show sbi card transactions",
    "how much did I spend on hdfc card",
    "hdfc card food expenses this month",
    "biggest expense on icici card",
    "sbi card transactions last month",
    "average spending on axis card",
    # Merchant + Card queries
    "my swiggy transactions in sbi card",
    "uber rides on hdfc card",
    "amazon orders from my icici card",
    "total zomato spending on hdfc card",
    # Merchant + Date + Card queries
    "my swiggy transactions in january with sbi card",
    "zomato orders last month on hdfc card",
    "uber rides this week from icici card",
    "how much on swiggy in january from sbi card",
]

for q in batch_questions:
    ask(q)

---
## Intent accuracy on training data

How well does the model classify intents it was trained on? This reveals overfitting vs underfitting.

In [ ]:
correct = 0
total = 0
per_intent = {name: {"correct": 0, "total": 0, "low_conf": []} for name in intent_labels}
misclassified = []

for expected_intent, examples in INTENT_EXAMPLES.items():
    for text in examples:
        predicted, conf, _ = predict_intent(text)
        total += 1
        per_intent[expected_intent]["total"] += 1

        if predicted == expected_intent:
            correct += 1
            per_intent[expected_intent]["correct"] += 1
            if conf < 0.7:
                per_intent[expected_intent]["low_conf"].append((text, conf))
        else:
            misclassified.append((text, expected_intent, predicted, conf))

print(f"Overall intent accuracy: {correct}/{total} ({correct/total:.1%})\n")
print(f"{'Intent':<25} {'Acc':>6}  {'Correct':>7} / {'Total':>5}")
print("─" * 50)
for name in intent_labels:
    d = per_intent[name]
    acc = d['correct'] / d['total'] if d['total'] > 0 else 0
    marker = "  ⚠️" if acc < 0.9 else ""
    print(f"{name:<25} {acc:>5.0%}   {d['correct']:>5} / {d['total']:>5}{marker}")

if misclassified:
    print(f"\n❌ Misclassified ({len(misclassified)}):")
    for text, expected, predicted, conf in misclassified:
        print(f"  \"{text}\"")
        print(f"    expected: {expected} → got: {predicted} ({conf:.0%})")

# Low confidence correct predictions
all_low = [(name, t, c) for name in intent_labels for t, c in per_intent[name]["low_conf"]]
if all_low:
    print(f"\n⚠️  Low confidence (<70%) but correct ({len(all_low)}):")
    for name, text, conf in sorted(all_low, key=lambda x: x[2]):
        print(f"  [{name}] \"{text}\" ({conf:.0%})")

Overall intent accuracy: 163/175 (93.1%)

Intent                       Acc  Correct / Total
──────────────────────────────────────────────────
average_spend               93%      14 /    15
category_spend              70%      14 /    20  ⚠️
count_transactions         100%      30 /    30
highest_transaction         87%      13 /    15  ⚠️
list_transactions           95%      19 /    20
lowest_transaction         100%      10 /    10
monthly_summary             87%      13 /    15  ⚠️
total_spent                100%      30 /    30
transactions_on_date       100%      20 /    20

❌ Misclassified (12):
  "how much did I spend on food and dining"
    expected: category_spend → got: total_spent (80%)
  "what is my shopping expense"
    expected: category_spend → got: average_spend (42%)
  "utility bill total"
    expected: category_spend → got: total_spent (53%)
  "travel expenses this month"
    expected: category_spend → got: monthly_summary (29%)
  "which category did I spend most on"

---
## Confidence distribution

Visualize how confident the model is across all training examples.

In [ ]:
import matplotlib.pyplot as plt

confidences = []
for examples in INTENT_EXAMPLES.values():
    for text in examples:
        _, conf, _ = predict_intent(text)
        confidences.append(conf)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Histogram
axes[0].hist(confidences, bins=20, color="#00E5A0", edgecolor="#0A0E1A", alpha=0.85)
axes[0].axvline(x=0.7, color="red", linestyle="--", label="70% threshold")
axes[0].set_xlabel("Confidence")
axes[0].set_ylabel("Count")
axes[0].set_title("Intent Confidence Distribution")
axes[0].legend()

# Per-intent box plot
intent_confs = {}
for intent_name, examples in INTENT_EXAMPLES.items():
    intent_confs[intent_name] = []
    for text in examples:
        _, conf, _ = predict_intent(text)
        intent_confs[intent_name].append(conf)

names = list(intent_confs.keys())
data = [intent_confs[n] for n in names]
bp = axes[1].boxplot(data, labels=[n.replace("_", "\n") for n in names], patch_artist=True)
for patch in bp["boxes"]:
    patch.set_facecolor("#00E5A0")
    patch.set_alpha(0.7)
axes[1].set_ylabel("Confidence")
axes[1].set_title("Confidence by Intent")
axes[1].tick_params(axis="x", rotation=45)
axes[1].axhline(y=0.7, color="red", linestyle="--", alpha=0.5)

plt.tight_layout()
plt.show()

below_70 = sum(1 for c in confidences if c < 0.7)
print(f"\n{below_70}/{len(confidences)} predictions below 70% confidence")

ModuleNotFoundError: No module named 'matplotlib'

---
## Confusion matrix

See which intents the model confuses with each other.

In [ ]:
n = len(intent_labels)
confusion = np.zeros((n, n), dtype=int)
label_to_idx = {name: i for i, name in enumerate(intent_labels)}

for expected_intent, examples in INTENT_EXAMPLES.items():
    for text in examples:
        predicted, _, _ = predict_intent(text)
        confusion[label_to_idx[expected_intent]][label_to_idx[predicted]] += 1

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(confusion, cmap="Greens")

short_labels = [l.replace("_", "\n") for l in intent_labels]
ax.set_xticks(range(n))
ax.set_yticks(range(n))
ax.set_xticklabels(short_labels, rotation=45, ha="right", fontsize=9)
ax.set_yticklabels(short_labels, fontsize=9)
ax.set_xlabel("Predicted")
ax.set_ylabel("Actual")
ax.set_title("Intent Confusion Matrix")

# Annotate cells
for i in range(n):
    for j in range(n):
        if confusion[i][j] > 0:
            color = "white" if confusion[i][j] > confusion.max() * 0.5 else "black"
            ax.text(j, i, str(confusion[i][j]), ha="center", va="center", color=color, fontsize=10)

plt.colorbar(im)
plt.tight_layout()
plt.show()

---
## Entity extraction accuracy

Test entity model against hand-annotated examples from training data.

In [ ]:
from training_data import ENTITY_EXAMPLES

entity_correct = 0
entity_total = 0
entity_errors = []
per_type = {"MERCHANT": {"tp": 0, "fp": 0, "fn": 0},
            "CATEGORY": {"tp": 0, "fp": 0, "fn": 0},
            "DATE":     {"tp": 0, "fp": 0, "fn": 0},
            "AMOUNT":   {"tp": 0, "fp": 0, "fn": 0},
            "CARD":     {"tp": 0, "fp": 0, "fn": 0}}

for text, char_entities in ENTITY_EXAMPLES:
    predicted_entities, _ = predict_entities(text)

    # Build expected entities from char spans
    expected = {}
    words = text.lower().split()
    for start, end, etype in char_entities:
        val = text[start:end].lower()
        expected[etype.lower()] = val

    entity_total += 1
    match = True

    # Check each expected entity type
    all_types = set(list(expected.keys()) + list(predicted_entities.keys()))
    for etype in all_types:
        etype_upper = etype.upper()
        if etype_upper not in per_type:
            continue
        exp_val = expected.get(etype)
        pred_val = predicted_entities.get(etype)

        if exp_val and pred_val:
            # Check if predicted contains the expected value (fuzzy match)
            if exp_val in pred_val or pred_val in exp_val:
                per_type[etype_upper]["tp"] += 1
            else:
                per_type[etype_upper]["fp"] += 1
                per_type[etype_upper]["fn"] += 1
                match = False
        elif pred_val and not exp_val:
            per_type[etype_upper]["fp"] += 1
            match = False
        elif exp_val and not pred_val:
            per_type[etype_upper]["fn"] += 1
            match = False

    if match:
        entity_correct += 1
    else:
        entity_errors.append((text, expected, predicted_entities))

print(f"Entity extraction: {entity_correct}/{entity_total} examples fully correct ({entity_correct/entity_total:.1%})\n")

print(f"{'Entity Type':<15} {'Precision':>10} {'Recall':>10} {'F1':>10}  {'TP':>4} {'FP':>4} {'FN':>4}")
print("─" * 65)
for etype in ["MERCHANT", "CATEGORY", "DATE", "AMOUNT", "CARD"]:
    d = per_type[etype]
    prec = d["tp"] / (d["tp"] + d["fp"]) if (d["tp"] + d["fp"]) > 0 else 0
    rec = d["tp"] / (d["tp"] + d["fn"]) if (d["tp"] + d["fn"]) > 0 else 0
    f1 = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0
    print(f"{etype:<15} {prec:>9.0%} {rec:>9.0%} {f1:>9.0%}  {d['tp']:>4} {d['fp']:>4} {d['fn']:>4}")

if entity_errors:
    print(f"\n❌ Errors ({len(entity_errors)}):")
    for text, expected, predicted in entity_errors[:15]:
        print(f"  \"{text}\"")
        print(f"    expected: {expected}")
        print(f"    got:      {predicted}")

---
## Interactive loop

Keep asking questions without editing cells. Run this cell and type questions one at a time.

In [ ]:
print("Type a question and press Enter. Type 'quit' to stop.\n")

while True:
    try:
        q = input("Ask: ").strip()
        if not q or q.lower() in ("quit", "exit", "q"):
            print("Done!")
            break
        ask(q)
    except (KeyboardInterrupt, EOFError):
        print("\nDone!")
        break

---
## Stress test: unseen queries

These are questions NOT in the training data — tests real generalization.

In [ ]:
unseen_queries = [
    # Rephrased / novel
    ("how many times I ate out", "count_transactions"),
    ("total money gone on rides", "total_spent"),
    ("what purchases did I make recently", "list_transactions"),
    ("my most expensive buy", "highest_transaction"),
    ("how much do I typically spend", "average_spend"),
    ("break it down by month", "monthly_summary"),
    ("spending overview by category", "category_spend"),
    ("what did I buy two days ago", "transactions_on_date"),
    # Informal / short
    ("swiggy total", "total_spent"),
    ("biggest purchase", "highest_transaction"),
    ("last month food", "category_spend"),
    ("count uber", "count_transactions"),
    # Tricky / ambiguous
    ("where is my money going", "category_spend"),
    ("am I spending too much on food", "category_spend"),
    ("show me everything from amazon", "list_transactions"),
    ("netflix and spotify charges", "list_transactions"),
    # Card queries (unseen)
    ("show all transactions on my sbi card", "list_transactions"),
    ("how much on icici card this month", "total_spent"),
    ("kotak card biggest purchase", "highest_transaction"),
    ("axis card monthly breakdown", "monthly_summary"),
]

correct = 0
print(f"{'Question':<45} {'Expected':<22} {'Got':<22} {'Conf':>6} {'OK?'}")
print("─" * 105)

for question, expected in unseen_queries:
    predicted, conf, _ = predict_intent(question)
    ok = predicted == expected
    if ok:
        correct += 1
    mark = "✅" if ok else "❌"
    print(f"{question:<45} {expected:<22} {predicted:<22} {conf:>5.0%} {mark}")

print(f"\nUnseen accuracy: {correct}/{len(unseen_queries)} ({correct/len(unseen_queries):.0%})")
print("Low unseen accuracy → model may need more diverse training data.")

---
## Model metadata

In [ ]:
import os

intent_size = os.path.getsize(f"{OUTPUT_DIR}/intent_model.tflite") / 1024
entity_size = os.path.getsize(f"{OUTPUT_DIR}/entity_model.tflite") / 1024
total_training = sum(len(v) for v in INTENT_EXAMPLES.values())

print(f"Intent model:  {intent_size:.1f} KB  |  {len(intent_labels)} intents  |  {len(intent_vocab)} vocab")
print(f"Entity model:  {entity_size:.1f} KB  |  {len(entity_labels)} tags      |  {len(entity_vocab)} vocab")
print(f"Training data: {total_training} intent examples, {len(ENTITY_EXAMPLES)} entity examples")
print(f"Sequence len:  {SEQUENCE_LENGTH} tokens")
print(f"Combined size: {intent_size + entity_size:.1f} KB (on-device)")